In [66]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import torch
import bitsandbytes
from PIL import Image
from IPython.display import display
from pathlib import Path
import unicodedata
import sys
import torch
import open_clip
import chromadb
import re
import os
import numpy as np
import pandas as pd
import subprocess
from PIL import Image

import whisper
from jiwer import wer, cer
import asyncio
import edge_tts
from pocket_tts import TTSModel
from tqdm.auto import tqdm
import librosa
from speechmos import dnsmos

import scipy.io.wavfile
from scipy.io import wavfile
import soundfile as sf
from datetime import timedelta
from sqlalchemy.orm import Session
from sqlalchemy import text
import json
import boto3
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor
import pillow_avif
import shutil
import hashlib

In [67]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine
from src.models import Pharaoh

In [68]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(torch.version.cuda)
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.7.1+cu118
11.8
CUDA available: True


In [69]:
!nvidia-smi

Sat Apr  4 01:29:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   37C    P8              2W /   70W |    5512MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [70]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [71]:
NAME = "Ramesses II"
is_landmark = False
if is_landmark:
    with Session(engine) as session:
        result = session.execute(text("SELECT landmark_script FROM landmarks_scripts as ls, landmarks as l WHERE ls.landmark_id = l.id AND l.name = :name"), {"name": NAME})
        script = result.fetchall()[0][0]

else:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": NAME})
        script = result.    fetchall()[0][0]

print(script)

Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.

Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.

He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers.

Ruling 

In [72]:
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
paragraphs

['Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.',
 'Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.',
 "He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers."

In [73]:
sentences = []
sentences_per_paragraph = []
for p in paragraphs:
    sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
    sentences_per_paragraph.append(len(sentences[-1]))
sentences

[['Rameses II was one of ancient Egypt’s greatest pharaohs.',
  'Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.',
  'At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.',
  'He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.'],
 ['Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.',
  'This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.',
  'Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.'],
 ["He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength.",
  'Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peac

### Audio Evaluation

In [75]:
def get_script_by_name(name, is_landmark=False):
    with Session(engine) as session:
        if is_landmark:
            result = session.execute(
                text("""
                    SELECT landmark_script
                    FROM landmarks_scripts AS ls, landmarks AS l
                    WHERE ls.landmark_id = l.id AND l.name = :name
                """),
                {"name": name}
            )
        else:
            result = session.execute(
                text("""
                    SELECT pharaoh_script
                    FROM pharaohs_scripts AS ps, pharaohs AS p
                    WHERE ps.pharaoh_id = p.id AND p.name = :name
                """),
                {"name": name}
            )

        rows = result.fetchall()

    if not rows:
        return None

    return rows[0][0]

In [76]:
def get_pharaoh_names():
    with Session(engine) as session:
        result = session.execute(
            text("""
                SELECT p.name
                FROM pharaohs_scripts AS ps, pharaohs AS p
                WHERE p.id = ps.pharaoh_id
            """)
        )
        names = [row[0] for row in result.fetchall()]

    names_to_remove = [
        'Amenhotep III and God Re-Horakhty',
        'Amenhotep III and Wife Queen Tiye',
        'Amun and Mut (God & Goddess)',
        'Hor (son of Ankh-Khonsu)',
        'Itysen (Prince, probably son of Djedefre)',
        'Isis (Goddess) with her child',
        'Menkaure and his wife (likely Queen Khamerernebty II)',
        'Menkaure with Goddesses Hathor and Bat',
        'Merenptah and Goddess Mut',
        'Ramesses II and God Ptah and Goddess Sekhment',
        'Nectanebo II and God Horus',
        'Osirian Triad and a King (Osiris, Isis, Horus, and Hormheb)',
        'Rameses II and Hauron',
        'Ramesess II and God Amun & Goddess Mut',
        'Ramesses II and God Ptah',
        'Ramesses II and God Ptah and a Goddess',
        'Ramesses II and Goddess',
        'Ramesses II and Goddess Anath',
        'Ramesses II and Goddesses Isis and Hathor',
        'Ramesses III and Gods Horus and Seth',
    ]

    names = [n for n in names if n not in names_to_remove]
    return names

In [77]:
names = get_pharaoh_names()
print("Total names:", len(names))

Total names: 80


In [78]:
# Create evaluation output folder (will have folder for each model)
BASE_EVAL_DIR = Path("Outputs") / "audio_eval"
BASE_EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Evaluation folder:", BASE_EVAL_DIR.resolve())

Evaluation folder: D:\GP\ECHO\experiments\video_generation\video_generation_pharaohs\Outputs\audio_eval


In [79]:
def safe_name_for_path(name):
    return (
        name.replace("/", "_")
            .replace("\\", "_")
            .replace(":", "_")
            .replace("*", "_")
            .replace("?", "_")
            .replace('"', "_")
            .replace("<", "_")
            .replace(">", "_")
            .replace("|", "_")
            .strip()
    )

def get_script_model_output_dir(model_name, script_name):
    safe_script_name = safe_name_for_path(script_name)
    model_dir = BASE_EVAL_DIR / model_name / safe_script_name
    model_dir.mkdir(parents=True, exist_ok=True)
    return model_dir

In [80]:
def split_into_sentences(script):
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
    sentences = []
    for p in paragraphs:
        sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
    return paragraphs, sentences

In [81]:
def build_sentence_table(paragraphs, sentences):
    sentence_rows = []

    for para_id, para_sentences in enumerate(sentences):    
        for sent_id, sent_text in enumerate(para_sentences):
            sentence_rows.append({
                "paragraph_id": para_id,
                "sentence_id": sent_id,
                "sentence_global_id": len(sentence_rows),
                "reference_text": sent_text
            })

    sentence_df = pd.DataFrame(sentence_rows)
    return sentence_df


In [82]:
def build_sentence_filename(paragraph_id, sentence_id):
    return f"p{paragraph_id:03d}_s{sentence_id:03d}.wav"


In [83]:
def generate_pocket_tts_audios(sent_df, script_name, voice_name="alba"):
    model_dir = get_script_model_output_dir("pocket_tts", script_name)

    tts_model = TTSModel.load_model()
    voice_state = tts_model.get_state_for_audio_prompt(voice_name)

    audio_paths = []
    generation_times = []

    for _, row in tqdm(sent_df.iterrows(), total=len(sent_df), desc=f"Generating pocket TTS - {script_name}"):
        text = row["reference_text"]
        file_name = row["file_name"]
        out_path = model_dir / file_name

        start_t = time.perf_counter()

        audio = tts_model.generate_audio(voice_state, text)

        scipy.io.wavfile.write(
            str(out_path),
            tts_model.sample_rate,
            audio.numpy()
        )

        end_t = time.perf_counter()

        audio_paths.append(str(out_path))
        generation_times.append(end_t - start_t)


    sent_df = sent_df.copy()
    sent_df["pocket_tts_audio_path"] = audio_paths
    sent_df["pocket_tts_generation_time_sec"] = generation_times

    return sent_df

In [84]:
sentt_df = build_sentence_table(paragraphs, sentences)

In [87]:
sentt_df["file_name"] = sentt_df.apply(
            lambda row: build_sentence_filename(row["paragraph_id"], row["sentence_id"]),
            axis=1
)

In [88]:
sentt_df

,paragraph_id,sentence_id,sentence_global_id,reference_text,file_name
0,0,0,0,Rameses II was one of ancient Egypt’s greatest...,p000_s000.wav
1,0,1,1,"Born to Seti I and Queen Tuya, he accompanied ...",p000_s001.wav
2,0,2,2,"At twenty-two, Rameses launched a Nubian campa...",p000_s002.wav
3,0,3,3,"He also fought wars in Canaan, bringing back t...",p000_s003.wav
4,1,0,4,Rameses built the grand capital of Per-Ramesse...,p001_s000.wav
5,1,1,5,"This city served as a palace, administrative c...",p001_s001.wav
6,1,2,6,"Its walls were adorned with faience tiles, sta...",p001_s002.wav
7,2,0,7,He continued his father’s restoration projects...,p002_s000.wav
8,2,1,8,Rameses faced a legendary battle against the H...,p002_s001.wav
9,3,0,9,Ruling for nearly sixty-six years until about ...,p003_s000.wav


In [89]:
sentt_df = generate_pocket_tts_audios(sentt_df,script_name=NAME,voice_name="alba")
sentt_df.head()

Generating pocket TTS - Ramesses II:   0%|          | 0/12 [00:00<?, ?it/s]

,paragraph_id,sentence_id,sentence_global_id,reference_text,file_name,pocket_tts_audio_path,pocket_tts_generation_time_sec
0,0,0,0,Rameses II was one of ancient Egypt’s greatest...,p000_s000.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,1.909649
1,0,1,1,"Born to Seti I and Queen Tuya, he accompanied ...",p000_s001.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,2.928146
2,0,2,2,"At twenty-two, Rameses launched a Nubian campa...",p000_s002.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,2.096416
3,0,3,3,"He also fought wars in Canaan, bringing back t...",p000_s003.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,1.650421
4,1,0,4,Rameses built the grand capital of Per-Ramesse...,p001_s000.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p001...,1.699344


In [90]:
from IPython.display import Audio, display

sample_path = sentt_df.loc[2, "pocket_tts_audio_path"]
print("Text:", sentt_df.loc[2, "reference_text"])
display(Audio(sample_path))

Text: At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.


In [17]:
async def edge_tts_save(text, out_path, voice="en-US-ChristopherNeural", rate="+0%"):
    communicate = edge_tts.Communicate(
        text=text,
        voice=voice,
        rate=rate
    )
    await communicate.save(str(out_path))

async def generate_edge_tts_audios(sent_df, script_name, voice="en-US-ChristopherNeural", rate="+0%"):
    model_dir = get_script_model_output_dir("edge_tts", script_name)

    audio_paths = []
    generation_times = []

    for _, row in tqdm(sent_df.iterrows(), total=len(sent_df), desc=f"Generating Edge-TTS - {script_name}"):
        text = row["reference_text"]
        file_name = row["file_name"]
        out_path = model_dir / file_name

        start_t = time.perf_counter()

        await edge_tts_save(text, out_path, voice=voice, rate=rate)

        end_t = time.perf_counter()
      

        audio_paths.append(str(out_path))
        generation_times.append(end_t - start_t)

    sent_df = sent_df.copy()
    sent_df["edge_tts_audio_path"] = audio_paths
    sent_df["edge_tts_generation_time_sec"] = generation_times

    return sent_df

In [91]:
sentt_df = await generate_edge_tts_audios(sentt_df,script_name=NAME,voice="en-US-ChristopherNeural",rate="+0%")
sentt_df.head()

Generating Edge-TTS - Ramesses II:   0%|          | 0/12 [00:00<?, ?it/s]

,paragraph_id,sentence_id,sentence_global_id,reference_text,file_name,pocket_tts_audio_path,pocket_tts_generation_time_sec,edge_tts_audio_path,edge_tts_generation_time_sec
0,0,0,0,Rameses II was one of ancient Egypt’s greatest...,p000_s000.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,1.909649,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.754693
1,0,1,1,"Born to Seti I and Queen Tuya, he accompanied ...",p000_s001.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,2.928146,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.609418
2,0,2,2,"At twenty-two, Rameses launched a Nubian campa...",p000_s002.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,2.096416,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.656488
3,0,3,3,"He also fought wars in Canaan, bringing back t...",p000_s003.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,1.650421,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.620548
4,1,0,4,Rameses built the grand capital of Per-Ramesse...,p001_s000.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p001...,1.699344,Outputs\audio_eval\edge_tts\Ramesses II\p001_s...,0.597224


In [92]:
from IPython.display import Audio, display

sample_path = sentt_df.loc[2, "edge_tts_audio_path"]
print("Text:", sentt_df.loc[2, "reference_text"])
display(Audio(sample_path))

Text: At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.


In [18]:
def count_words(script):
    return len(re.findall(r"\b[\w']+\b", script))

def get_audio_duration_sec(audio_path):
    data, sr = sf.read(audio_path)
    return len(data) / sr

# Words per minute calculation
def compute_wpm(word_count, duration_sec):
    if duration_sec <= 0:   # to avoid division errors
        return np.nan
    return (word_count / duration_sec) * 60

In [19]:
def add_basic_audio_metrics(sent_df, audio_path_col, prefix):
    durations = []
    word_counts = []
    wpms = []

    for _, row in sent_df.iterrows():
        text = row["reference_text"]
        audio_path = row[audio_path_col]

        word_count = count_words(text)
        duration_sec = get_audio_duration_sec(audio_path)
        wpm = compute_wpm(word_count, duration_sec)

        word_counts.append(word_count)
        durations.append(duration_sec)
        wpms.append(wpm)

    sent_df = sent_df.copy()
    sent_df[f"{prefix}_word_count"] = word_counts
    sent_df[f"{prefix}_duration_sec"] = durations
    sent_df[f"{prefix}_wpm"] = wpms

    return sent_df

In [20]:
# Before comparing reference text and Whisper text, normalize both (to have same format)
def normalize_text_for_eval(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [21]:
whisper_model = whisper.load_model("small")
def transcribe_with_whisper(audio_path):
    result = whisper_model.transcribe(audio_path, fp16=False)
    return result["text"].strip()

In [22]:
def evaluate_transcription_metrics(audio_path, reference_text):
    hypothesis_text = transcribe_with_whisper(audio_path)

    ref_norm = normalize_text_for_eval(reference_text)
    hyp_norm = normalize_text_for_eval(hypothesis_text)

    return {
        "hypothesis_text": hypothesis_text,
        "reference_normalized": ref_norm,
        "hypothesis_normalized": hyp_norm,
        "wer": wer(ref_norm, hyp_norm),   # word error rate
    }

In [23]:
def add_transcription_metrics(sent_df, audio_path_col, prefix):
    hypothesis_texts = []
    ref_norms = []
    hyp_norms = []
    wers = []

    for _, row in sent_df.iterrows():
        audio_path = row[audio_path_col]
        reference_text = row["reference_text"]

        result = evaluate_transcription_metrics(audio_path, reference_text)

        hypothesis_texts.append(result["hypothesis_text"])
        ref_norms.append(result["reference_normalized"])
        hyp_norms.append(result["hypothesis_normalized"])
        wers.append(result["wer"])

    sent_df = sent_df.copy()
    sent_df[f"{prefix}_hypothesis_text"] = hypothesis_texts
    sent_df[f"{prefix}_reference_normalized"] = ref_norms
    sent_df[f"{prefix}_hypothesis_normalized"] = hyp_norms
    sent_df[f"{prefix}_wer"] = wers

    return sent_df

In [24]:
# Audio Quality Metrics
def compute_silence_ratio(audio_path, top_db=30):
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    if len(y) == 0:
        return np.nan

    non_silent_intervals = librosa.effects.split(y, top_db=top_db)

    non_silent_samples = sum(end - start for start, end in non_silent_intervals)
    total_samples = len(y)

    return 1.0 - (non_silent_samples / total_samples)


In [25]:
def add_quality_metric(sent_df, audio_path_col, prefix, silence_top_db=30):
    silence_ratios = []

    for _, row in sent_df.iterrows():
        audio_path = row[audio_path_col]
        silence_ratio = compute_silence_ratio(audio_path, top_db=silence_top_db)
        silence_ratios.append(silence_ratio)

    sent_df = sent_df.copy()
    sent_df[f"{prefix}_silence_ratio"] = silence_ratios
    return sent_df

##### Overall 

In [26]:
async def evaluate_one_entity(name, is_landmark=False):
    print(f"\nEvaluating: {name}")

    # 1) fetch script
    script = get_script_by_name(name, is_landmark=is_landmark)

    if script is None or not str(script).strip():
        print(f"Skipped {name}: no script found")
        return None

    # 2) build sentence table using your existing functions
    paragraphs, sentence_lists = split_into_sentences(script)
    sent_df = build_sentence_table(paragraphs, sentence_lists)

    # 3) attach script-level metadata
    sent_df = sent_df.copy()
    sent_df["entity_name"] = name
    sent_df["is_landmark"] = is_landmark

    # 4) add file names if not already present
    if "file_name" not in sent_df.columns:
        sent_df["file_name"] = sent_df.apply(
            lambda row: build_sentence_filename(row["paragraph_id"], row["sentence_id"]),
            axis=1
        )

    # 5) generate pocket model audios + generation times
    sent_df = generate_pocket_tts_audios(
        sent_df,
        script_name=name,
        voice_name="alba"
    )

    # 6) generate edge model audios + generation times
    sent_df = await generate_edge_tts_audios(
        sent_df,
        script_name=name,
        voice="en-US-ChristopherNeural",
        rate="+0%"
    )

    # 7) add basic metrics (WPM)
    sent_df = add_basic_audio_metrics(
        sent_df,
        audio_path_col="pocket_tts_audio_path",
        prefix="pocket_tts"
    )
    sent_df = add_basic_audio_metrics(
        sent_df,
        audio_path_col="edge_tts_audio_path",
        prefix="edge_tts"
    )

    # 8) add transcription metrics (WER)
    sent_df = add_transcription_metrics(
        sent_df,
        audio_path_col="pocket_tts_audio_path",
        prefix="pocket_tts"
    )
    sent_df = add_transcription_metrics(
        sent_df,
        audio_path_col="edge_tts_audio_path",
        prefix="edge_tts"
    )

    # 9) add quality metrics (Silence Ratio)
    sent_df = add_quality_metric(
        sent_df,
        audio_path_col="pocket_tts_audio_path",
        prefix="pocket_tts"
    )
    sent_df = add_quality_metric(
        sent_df,
        audio_path_col="edge_tts_audio_path",
        prefix="edge_tts"
    )

    return sent_df

In [ ]:
def summarize_entity_metrics(sent_df):
    entity_name = sent_df["entity_name"].iloc[0]

    pocket_word_total = sent_df["pocket_tts_word_count"].sum()
    edge_word_total = sent_df["edge_tts_word_count"].sum()
    edge_trimmed_word_total = sent_df["edge_tts_trimmed_word_count"].sum()

    pocket_duration_total = sent_df["pocket_tts_duration_sec"].sum()
    edge_duration_total = sent_df["edge_tts_duration_sec"].sum()
    edge_trimmed_duration_total = sent_df["edge_tts_trimmed_duration_sec"].sum()

    return pd.DataFrame([{
        "entity_name": entity_name,

        # Pocket-TTS
        "pocket_tts_wpm": (pocket_word_total / pocket_duration_total) * 60 if pocket_duration_total > 0 else np.nan,
        "pocket_tts_wer": (
            (sent_df["pocket_tts_wer"] * sent_df["pocket_tts_word_count"]).sum() / pocket_word_total
            if pocket_word_total > 0 else np.nan
        ),
        "pocket_tts_silence_ratio": sent_df["pocket_tts_silence_ratio"].mean(),
        "pocket_tts_generation_time_sec": sent_df["pocket_tts_generation_time_sec"].sum(),

        # Edge-TTS
        "edge_tts_wpm": (edge_word_total / edge_duration_total) * 60 if edge_duration_total > 0 else np.nan,
        "edge_tts_wer": (
            (sent_df["edge_tts_wer"] * sent_df["edge_tts_word_count"]).sum() / edge_word_total
            if edge_word_total > 0 else np.nan
        ),
        "edge_tts_silence_ratio": sent_df["edge_tts_silence_ratio"].mean(),
        "edge_tts_generation_time_sec": sent_df["edge_tts_generation_time_sec"].sum(),
    }])

In [26]:
names = get_pharaoh_names()

all_sent_dfs = []
entity_summary_dfs = []

batch_size = 10

for batch_start in range(0, len(names), batch_size):
    batch_names = names[batch_start: batch_start + batch_size]

    print(f"\n{'='*20}")
    print(f"Running batch {batch_start+1} to {batch_start + len(batch_names)}")
    print(f"{'='*20}")

    for i, name in enumerate(batch_names, start=batch_start + 1):
        print(f"\n===== {i}/{len(names)} : {name} =====")

        try:
            sent_df_entity = await evaluate_one_entity(name, is_landmark=False)

            if sent_df_entity is not None and len(sent_df_entity) > 0:
                all_sent_dfs.append(sent_df_entity)

                entity_summary = summarize_entity_metrics(sent_df_entity)
                entity_summary_dfs.append(entity_summary)

        except Exception as e:
            print(f"Failed on {name}: {type(e).__name__}: {e}")
            continue

    # save cumulative progress after each batch into the same files
    if all_sent_dfs:
        pd.concat(all_sent_dfs, ignore_index=True).to_csv(
            "Outputs/audio_eval/intermediate_sentence_results.csv",
            index=False
        )

    if entity_summary_dfs:
        pd.concat(entity_summary_dfs, ignore_index=True).to_csv(
            "Outputs/audio_eval/intermediate_entity_summary.csv",
            index=False
        )

    print(f"Saved cumulative results through entity {batch_start + len(batch_names)}")

# final save
if all_sent_dfs:
    all_sent_df = pd.concat(all_sent_dfs, ignore_index=True)
    all_sent_df.to_csv("Outputs/audio_eval/final_sentence_results.csv", index=False)
else:
    all_sent_df = pd.DataFrame()

if entity_summary_dfs:
    entity_summary_df = pd.concat(entity_summary_dfs, ignore_index=True)
    entity_summary_df.to_csv("Outputs/audio_eval/final_entity_summary.csv", index=False)
else:
    entity_summary_df = pd.DataFrame()

print("Done.")
print("Sentence-level rows:", len(all_sent_df))
print("Entity-level rows:", len(entity_summary_df))


Running batch 1 to 10

===== 1/80 : Achoris =====

Evaluating: Achoris


Generating pocket TTS - Achoris:   0%|          | 0/8 [00:00<?, ?it/s]

Generating Edge-TTS - Achoris:   0%|          | 0/8 [00:00<?, ?it/s]


===== 2/80 : Ahmose I =====

Evaluating: Ahmose I


Generating pocket TTS - Ahmose I:   0%|          | 0/7 [00:00<?, ?it/s]

...

Generating Edge-TTS - Tutankhamun:   0%|          | 0/8 [00:00<?, ?it/s]


===== 80/80 : Yuya (father of Queen Tiye) =====

Evaluating: Yuya (father of Queen Tiye)


Generating pocket TTS - Yuya (father of Queen Tiye):   0%|          | 0/11 [00:00<?, ?it/s]

Generating Edge-TTS - Yuya (father of Queen Tiye):   0%|          | 0/11 [00:00<?, ?it/s]

Saved cumulative results through entity 80
Done.
Sentence-level rows: 775
Entity-level rows: 79


In [27]:
missing_name = "Amenemhat I"

# 1) run only the missing entity
sent_df_entity = await evaluate_one_entity(missing_name, is_landmark=False)
entity_summary = summarize_entity_metrics(sent_df_entity)

# 2) read existing full results
all_sent_df_existing = pd.read_csv("Outputs/audio_eval/final_sentence_results.csv")
entity_summary_df_existing = pd.read_csv("Outputs/audio_eval/final_entity_summary.csv")

# 3) append
all_sent_df_updated = pd.concat(
    [all_sent_df_existing, sent_df_entity],
    ignore_index=True
)

entity_summary_df_updated = pd.concat(
    [entity_summary_df_existing, entity_summary],
    ignore_index=True
)

# 4) deduplicate
all_sent_df_updated = all_sent_df_updated.drop_duplicates(
    subset=["entity_name", "paragraph_id", "sentence_id"],
    keep="last"
).reset_index(drop=True)

entity_summary_df_updated = entity_summary_df_updated.drop_duplicates(
    subset=["entity_name"],
    keep="last"
).reset_index(drop=True)

# 5) save back
all_sent_df_updated.to_csv("Outputs/audio_eval/final_sentence_results.csv", index=False)
entity_summary_df_updated.to_csv("Outputs/audio_eval/final_entity_summary.csv", index=False)

# optional: keep intermediate files synced too
all_sent_df_updated.to_csv("Outputs/audio_eval/intermediate_sentence_results.csv", index=False)
entity_summary_df_updated.to_csv("Outputs/audio_eval/intermediate_entity_summary.csv", index=False)

print("Added missing entity successfully.")
print("Sentence-level rows:", len(all_sent_df_updated))
print("Entity-level rows:", len(entity_summary_df_updated))


Evaluating: Amenemhat I


Generating pocket TTS - Amenemhat I:   0%|          | 0/11 [00:00<?, ?it/s]

Generating Edge-TTS - Amenemhat I:   0%|          | 0/11 [00:00<?, ?it/s]

Added missing entity successfully.
Sentence-level rows: 786
Entity-level rows: 80


In [28]:
final_avg_across_entities = pd.DataFrame({
    "model": ["pocket_tts", "edge_tts"],
    "avg_wpm_across_80_entities": [
        entity_summary_df_updated["pocket_tts_wpm"].mean(),
        entity_summary_df_updated["edge_tts_wpm"].mean()
    ],
    "avg_wer_across_80_entities": [
        entity_summary_df_updated["pocket_tts_wer"].mean(),
        entity_summary_df_updated["edge_tts_wer"].mean()
    ],
    "avg_silence_ratio_across_80_entities": [
        entity_summary_df_updated["pocket_tts_silence_ratio"].mean(),
        entity_summary_df_updated["edge_tts_silence_ratio"].mean()
    ],
    "avg_generation_time_sec_across_80_entities": [
        entity_summary_df_updated["pocket_tts_generation_time_sec"].mean(),
        entity_summary_df_updated["edge_tts_generation_time_sec"].mean()
    ]
})

final_avg_across_entities

,model,avg_wpm_across_80_entities,avg_wer_across_80_entities,avg_silence_ratio_across_80_entities,avg_generation_time_sec_across_80_entities
0,pocket_tts,160.268440,0.097583,0.079502,29.283830
1,edge_tts,146.866449,0.099106,0.183899,10.031166


In [28]:
entity_summary_df_updated = pd.read_csv(
    "Outputs/audio_eval/final_entity_summary.csv"
)

all_sent_df_updated = pd.read_csv(
    "Outputs/audio_eval/final_sentence_results.csv"
)

In [29]:
final_avg_across_entities = pd.DataFrame({
    "model": ["pocket_tts", "edge_tts"],
    "avg_wpm_across_80_entities": [
        entity_summary_df_updated["pocket_tts_wpm"].mean(),
        entity_summary_df_updated["edge_tts_wpm"].mean()
    ],
    "avg_wer_across_80_entities": [
        entity_summary_df_updated["pocket_tts_wer"].mean(),
        entity_summary_df_updated["edge_tts_wer"].mean()
    ],
    "avg_silence_ratio_across_80_entities": [
        entity_summary_df_updated["pocket_tts_silence_ratio"].mean(),
        entity_summary_df_updated["edge_tts_silence_ratio"].mean()
    ],
    "avg_generation_time_sec_across_80_entities": [
        entity_summary_df_updated["pocket_tts_generation_time_sec"].mean(),
        entity_summary_df_updated["edge_tts_generation_time_sec"].mean()
    ]
})

final_avg_across_entities

,model,avg_wpm_across_80_entities,avg_wer_across_80_entities,avg_silence_ratio_across_80_entities,avg_generation_time_sec_across_80_entities
0,pocket_tts,160.268440,0.097583,0.079502,29.283830
1,edge_tts,146.866449,0.099106,0.183899,10.031166


In [ ]:
async def edge_tts_save(text, out_path, voice="en-US-ChristopherNeural", rate="+0%"):
    communicate = edge_tts.Communicate(
        text=text,
        voice=voice,
        rate=rate
    )
    await communicate.save(str(out_path))

def get_trimmed_audio_and_silence_stats(audio_path, top_db=20):
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    if len(y) == 0:
        return {
            "trimmed_audio": y,
            "sr": sr,
            "leading_silence_sec": 0.0,
            "trailing_silence_sec": 0.0,
            "total_outer_silence_sec": 0.0,
            "duration_before_trim_sec": 0.0,
            "duration_after_trim_sec": 0.0,
            "outer_silence_ratio": 0.0,
        }

    intervals = librosa.effects.split(y, top_db=top_db)

    duration_before = len(y) / sr

    if len(intervals) == 0:
        return {
            "trimmed_audio": y,
            "sr": sr,
            "leading_silence_sec": 0.0,
            "trailing_silence_sec": 0.0,
            "total_outer_silence_sec": 0.0,
            "duration_before_trim_sec": duration_before,
            "duration_after_trim_sec": duration_before,
            "outer_silence_ratio": 0.0,
        }

    first_start = intervals[0][0]
    last_end = intervals[-1][1]

    trimmed = y[first_start:last_end]

    leading_silence_sec = first_start / sr
    trailing_silence_sec = (len(y) - last_end) / sr
    total_outer_silence_sec = leading_silence_sec + trailing_silence_sec
    duration_after = len(trimmed) / sr

    outer_silence_ratio = (
        total_outer_silence_sec / duration_before if duration_before > 0 else 0.0
    )

    return {
        "trimmed_audio": trimmed,
        "sr": sr,
        "leading_silence_sec": leading_silence_sec,
        "trailing_silence_sec": trailing_silence_sec,
        "total_outer_silence_sec": total_outer_silence_sec,
        "duration_before_trim_sec": duration_before,
        "duration_after_trim_sec": duration_after,
        "outer_silence_ratio": outer_silence_ratio,
    }


In [43]:
async def generate_edge_tts_trimmed_audios(
    sent_df,
    script_name,
    voice="en-US-ChristopherNeural",
    rate="+0%",
    trim_top_db=20,
    keep_raw=False
):
    raw_dir = get_script_model_output_dir("edge_tts_trimmed_raw_temp", script_name)
    trimmed_dir = get_script_model_output_dir("edge_tts_trimmed", script_name)

    trimmed_paths = []
    raw_paths = []

    raw_generation_times = []
    trim_times = []
    total_pipeline_times = []

    leading_silences = []
    trailing_silences = []
    total_outer_silences = []
    duration_before_list = []
    duration_after_list = []
    outer_silence_ratios = []

    for _, row in tqdm(sent_df.iterrows(), total=len(sent_df), desc=f"Generating Edge-TTS trimmed - {script_name}"):
        text = row["reference_text"]
        file_name = row["file_name"]

        raw_path = raw_dir / file_name
        trimmed_path = trimmed_dir / file_name

        start_gen = time.perf_counter()
        await edge_tts_save(text, raw_path, voice=voice, rate=rate)
        end_gen = time.perf_counter()
        gen_time = end_gen - start_gen

        start_trim = time.perf_counter()
        result = get_trimmed_audio_and_silence_stats(str(raw_path), top_db=trim_top_db)
        sf.write(str(trimmed_path), result["trimmed_audio"], result["sr"])
        end_trim = time.perf_counter()
        trim_time = end_trim - start_trim

        trimmed_paths.append(str(trimmed_path))
        raw_paths.append(str(raw_path))

        raw_generation_times.append(gen_time)
        trim_times.append(trim_time)
        total_pipeline_times.append(gen_time + trim_time)

        leading_silences.append(result["leading_silence_sec"])
        trailing_silences.append(result["trailing_silence_sec"])
        total_outer_silences.append(result["total_outer_silence_sec"])
        duration_before_list.append(result["duration_before_trim_sec"])
        duration_after_list.append(result["duration_after_trim_sec"])
        outer_silence_ratios.append(result["outer_silence_ratio"])

        if not keep_raw:
            try:
                raw_path.unlink(missing_ok=True)
            except Exception:
                pass

    sent_df = sent_df.copy()

    if keep_raw:
        sent_df["edge_tts_trimmed_raw_temp_audio_path"] = raw_paths

    sent_df["edge_tts_trimmed_audio_path"] = trimmed_paths
    sent_df["edge_tts_trimmed_raw_generation_time_sec"] = raw_generation_times
    sent_df["edge_tts_trimmed_trim_time_sec"] = trim_times
    sent_df["edge_tts_trimmed_generation_time_sec"] = total_pipeline_times

    sent_df["edge_tts_leading_silence_sec"] = leading_silences
    sent_df["edge_tts_trailing_silence_sec"] = trailing_silences
    sent_df["edge_tts_total_outer_silence_sec"] = total_outer_silences
    sent_df["edge_tts_duration_before_trim_sec"] = duration_before_list
    sent_df["edge_tts_duration_after_trim_sec"] = duration_after_list
    sent_df["edge_tts_outer_silence_ratio"] = outer_silence_ratios

    return sent_df

In [93]:
sentt_df = await generate_edge_tts_trimmed_audios(sentt_df,script_name=NAME,voice="en-US-ChristopherNeural",rate="+0%")
sentt_df.head()

Generating Edge-TTS trimmed - Ramesses II:   0%|          | 0/12 [00:00<?, ?it/s]

,paragraph_id,sentence_id,sentence_global_id,reference_text,file_name,pocket_tts_audio_path,pocket_tts_generation_time_sec,edge_tts_audio_path,edge_tts_generation_time_sec,edge_tts_trimmed_audio_path,edge_tts_trimmed_raw_generation_time_sec,edge_tts_trimmed_trim_time_sec,edge_tts_trimmed_generation_time_sec,edge_tts_leading_silence_sec,edge_tts_trailing_silence_sec,edge_tts_total_outer_silence_sec,edge_tts_duration_before_trim_sec,edge_tts_duration_after_trim_sec,edge_tts_outer_silence_ratio
0,0,0,0,Rameses II was one of ancient Egypt’s greatest...,p000_s000.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,1.909649,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.754693,Outputs\audio_eval\edge_tts_trimmed\Ramesses I...,0.530631,0.033228,0.563859,0.192000,0.896000,1.088000,4.416,3.328000,0.246377
1,0,1,1,"Born to Seti I and Queen Tuya, he accompanied ...",p000_s001.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,2.928146,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.609418,Outputs\audio_eval\edge_tts_trimmed\Ramesses I...,0.719032,0.020557,0.739590,0.192000,0.968000,1.160000,9.480,8.320000,0.122363
2,0,2,2,"At twenty-two, Rameses launched a Nubian campa...",p000_s002.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,2.096416,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.656488,Outputs\audio_eval\edge_tts_trimmed\Ramesses I...,0.580638,0.022675,0.603313,0.170667,0.954667,1.125333,7.248,6.122667,0.155261
3,0,3,3,"He also fought wars in Canaan, bringing back t...",p000_s003.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p000...,1.650421,Outputs\audio_eval\edge_tts\Ramesses II\p000_s...,0.620548,Outputs\audio_eval\edge_tts_trimmed\Ramesses I...,0.609718,0.027801,0.637519,0.213333,1.040000,1.253333,5.712,4.458667,0.219421
4,1,0,4,Rameses built the grand capital of Per-Ramesse...,p001_s000.wav,Outputs\audio_eval\pocket_tts\Ramesses II\p001...,1.699344,Outputs\audio_eval\edge_tts\Ramesses II\p001_s...,0.597224,Outputs\audio_eval\edge_tts_trimmed\Ramesses I...,0.569719,0.015196,0.584915,0.192000,0.957333,1.149333,5.736,4.586667,0.200372


In [94]:
from IPython.display import Audio, display

sample_path = sentt_df.loc[2, "edge_tts_trimmed_audio_path"]
print("Text:", sentt_df.loc[2, "reference_text"])
display(Audio(sample_path))

Text: At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.


In [97]:
sentt_df[["pocket_tts_generation_time_sec", "edge_tts_generation_time_sec", "edge_tts_trimmed_generation_time_sec"]].sum()

pocket_tts_generation_time_sec          26.225177
edge_tts_generation_time_sec             7.707485
edge_tts_trimmed_generation_time_sec     7.690359
dtype: float64

In [44]:
async def evaluate_one_entity_trimmed(name, is_landmark=False):
    print(f"\nEvaluating trimmed Edge-TTS: {name}")

    script = get_script_by_name(name, is_landmark=is_landmark)

    if script is None or not str(script).strip():
        print(f"Skipped {name}: no script found")
        return None

    paragraphs, sentence_lists = split_into_sentences(script)
    sent_df = build_sentence_table(paragraphs, sentence_lists)

    sent_df = sent_df.copy()
    sent_df["entity_name"] = name
    sent_df["is_landmark"] = is_landmark

    if "file_name" not in sent_df.columns:
        sent_df["file_name"] = sent_df.apply(
            lambda row: build_sentence_filename(row["paragraph_id"], row["sentence_id"]),
            axis=1
        )

    sent_df = await generate_edge_tts_trimmed_audios(
        sent_df,
        script_name=name,
        voice="en-US-ChristopherNeural",
        rate="+0%",
        trim_top_db=20,
        keep_raw=False
    )

    sent_df = add_basic_audio_metrics(
        sent_df,
        audio_path_col="edge_tts_trimmed_audio_path",
        prefix="edge_tts_trimmed"
    )

    sent_df = add_transcription_metrics(
        sent_df,
        audio_path_col="edge_tts_trimmed_audio_path",
        prefix="edge_tts_trimmed"
    )

    sent_df = add_quality_metric(
        sent_df,
        audio_path_col="edge_tts_trimmed_audio_path",
        prefix="edge_tts_trimmed",
        silence_top_db=30
    )

    return sent_df

In [45]:
names = get_pharaoh_names()

all_trimmed_sent_dfs = []

for i, name in enumerate(names, start=1):
    print(f"\n===== {i}/{len(names)} : {name} =====")

    try:
        sent_df_entity_trimmed = await evaluate_one_entity_trimmed(name, is_landmark=False)

        if sent_df_entity_trimmed is not None and len(sent_df_entity_trimmed) > 0:
            all_trimmed_sent_dfs.append(sent_df_entity_trimmed)

    except Exception as e:
        print(f"Failed on {name}: {type(e).__name__}: {e}")
        continue

trimmed_sent_df = pd.concat(all_trimmed_sent_dfs, ignore_index=True)


===== 1/80 : Achoris =====

Evaluating trimmed Edge-TTS: Achoris


Generating Edge-TTS trimmed - Achoris:   0%|          | 0/8 [00:00<?, ?it/s]


===== 2/80 : Ahmose I =====

Evaluating trimmed Edge-TTS: Ahmose I


Generating Edge-TTS trimmed - Ahmose I:   0%|          | 0/7 [00:00<?, ?it/s]


===== 3/80 : Akhenaton =====

Evaluating trimmed Edge-TTS: Akhenaton


...

Generating Edge-TTS trimmed - Tiye (Queen, wife of Amenhotep III):   0%|          | 0/8 [00:00<?, ?it/s]

Failed on Tiye (Queen, wife of Amenhotep III): WSServerHandshakeError: 503, message='Invalid response status', url='wss://speech.platform.bing.com/consumer/speech/synthesize/readaloud/edge/v1?TrustedClientToken=6A5AA1D4EAFF4E9FB37E23D68491D6F4&ConnectionId=cfca52fc5aa047578cd1645e141d396c&Sec-MS-GEC=480CCA870645990C725DC1B72DB62C29BFF0D3365FABE7B95CCF40063258E92E&Sec-MS-GEC-Version=1-143.0.3650.75'

===== 79/80 : Tutankhamun =====

Evaluating trimmed Edge-TTS: Tutankhamun


Generating Edge-TTS trimmed - Tutankhamun:   0%|          | 0/8 [00:00<?, ?it/s]


===== 80/80 : Yuya (father of Queen Tiye) =====

Evaluating trimmed Edge-TTS: Yuya (father of Queen Tiye)


Generating Edge-TTS trimmed - Yuya (father of Queen Tiye):   0%|          | 0/11 [00:00<?, ?it/s]

In [46]:
missing_name = "Tiye (Queen, wife of Amenhotep III)"

sent_df_entity_trimmed = await evaluate_one_entity_trimmed(
    missing_name,
    is_landmark=False
)


Evaluating trimmed Edge-TTS: Tiye (Queen, wife of Amenhotep III)


Generating Edge-TTS trimmed - Tiye (Queen, wife of Amenhotep III):   0%|          | 0/8 [00:00<?, ?it/s]

In [47]:
trimmed_sent_df = pd.concat(
    [trimmed_sent_df, sent_df_entity_trimmed],
    ignore_index=True
)

In [48]:
trimmed_sent_df = trimmed_sent_df.drop_duplicates(
    subset=["entity_name", "paragraph_id", "sentence_id"],
    keep="last"
).reset_index(drop=True)

In [49]:
trimmed_sent_df["entity_name"].nunique()

80

In [50]:
trimmed_cols_to_add = [
    "entity_name",
    "paragraph_id",
    "sentence_id",

    "edge_tts_trimmed_audio_path",
    "edge_tts_trimmed_raw_generation_time_sec",
    "edge_tts_trimmed_trim_time_sec",
    "edge_tts_trimmed_generation_time_sec",

    "edge_tts_trimmed_word_count",
    "edge_tts_trimmed_duration_sec",
    "edge_tts_trimmed_wpm",
    "edge_tts_trimmed_hypothesis_text",
    "edge_tts_trimmed_reference_normalized",
    "edge_tts_trimmed_hypothesis_normalized",
    "edge_tts_trimmed_wer",
    "edge_tts_trimmed_silence_ratio",

    "edge_tts_leading_silence_sec",
    "edge_tts_trailing_silence_sec",
    "edge_tts_total_outer_silence_sec",
    "edge_tts_duration_before_trim_sec",
    "edge_tts_duration_after_trim_sec",
    "edge_tts_outer_silence_ratio",
]

trimmed_sent_df = trimmed_sent_df[trimmed_cols_to_add].copy()

In [51]:
all_sent_df_updated = all_sent_df_updated.merge(
    trimmed_sent_df,
    on=["entity_name", "paragraph_id", "sentence_id"],
    how="left"
)

In [55]:
# 1) drop all old duplicate columns (_x)
cols_to_drop = [col for col in all_sent_df_updated.columns if col.endswith("_x")]
all_sent_df_updated = all_sent_df_updated.drop(columns=cols_to_drop)

# 2) rename _y → clean names
all_sent_df_updated.columns = [
    col.replace("_y", "") for col in all_sent_df_updated.columns
]

In [56]:
all_sent_df_updated.columns.to_list()

['paragraph_id',
 'sentence_id',
 'sentence_global_id',
 'reference_text',
 'entity_name',
 'is_landmark',
 'file_name',
 'pocket_tts_audio_path',
 'pocket_tts_generation_time_sec',
 'edge_tts_audio_path',
 'edge_tts_generation_time_sec',
 'pocket_tts_word_count',
 'pocket_tts_duration_sec',
 'pocket_tts_wpm',
 'edge_tts_word_count',
 'edge_tts_duration_sec',
 'edge_tts_wpm',
 'pocket_tts_hypothesis_text',
 'pocket_tts_reference_normalized',
 'pocket_tts_hypothesis_normalized',
 'pocket_tts_wer',
 'edge_tts_hypothesis_text',
 'edge_tts_reference_normalized',
 'edge_tts_hypothesis_normalized',
 'edge_tts_wer',
 'pocket_tts_silence_ratio',
 'edge_tts_silence_ratio',
 'edge_tts_trimmed_audio_path',
 'edge_tts_trimmed_raw_generation_time_sec',
 'edge_tts_trimmed_trim_time_sec',
 'edge_tts_trimmed_generation_time_sec',
 'edge_tts_trimmed_word_count',
 'edge_tts_trimmed_duration_sec',
 'edge_tts_trimmed_wpm',
 'edge_tts_trimmed_hypothesis_text',
 'edge_tts_trimmed_reference_normalized',
 'ed

In [57]:
all_sent_df_updated[[
    "edge_tts_leading_silence_sec",
    "edge_tts_trailing_silence_sec",
    "edge_tts_total_outer_silence_sec",
    "edge_tts_outer_silence_ratio"
]].describe()

,edge_tts_leading_silence_sec,edge_tts_trailing_silence_sec,edge_tts_total_outer_silence_sec,edge_tts_outer_silence_ratio
count,786.000000,786.000000,786.000000,786.000000
mean,0.197265,0.942511,1.139776,0.149977
std,0.019234,0.050135,0.054178,0.053975
min,0.170667,0.856000,1.026667,0.057481
25%,0.192000,0.902000,1.101333,0.115248
50%,0.192000,0.944000,1.138667,0.139743
75%,0.213333,0.965333,1.168000,0.170591
max,0.298667,1.168000,1.397333,0.541520


In [58]:
all_sent_df_updated[[
    "edge_tts_wpm",
    "edge_tts_trimmed_wpm",
    "edge_tts_wer",
    "edge_tts_trimmed_wer",
    "edge_tts_silence_ratio",
    "edge_tts_trimmed_silence_ratio"
]].describe()

,edge_tts_wpm,edge_tts_trimmed_wpm,edge_tts_wer,edge_tts_trimmed_wer,edge_tts_silence_ratio,edge_tts_trimmed_silence_ratio
count,786.000000,786.000000,786.000000,786.000000,786.000000,786.000000
mean,145.676127,171.664852,0.099996,0.099545,0.185578,0.050701
std,19.111213,22.679549,0.100424,0.098908,0.046090,0.035640
min,25.510204,54.086538,0.000000,0.000000,0.099678,0.000000
25%,133.132791,155.856423,0.000000,0.000000,0.153913,0.028037
50%,146.713615,172.002879,0.083333,0.083333,0.178140,0.046898
75%,159.214550,185.883621,0.153846,0.153846,0.208322,0.071503
max,216.894977,281.250000,0.833333,0.833333,0.541520,0.208417


In [63]:
all_sent_df_updated[[
    "edge_tts_generation_time_sec",
    "edge_tts_trimmed_raw_generation_time_sec",
    "edge_tts_trimmed_trim_time_sec",
    "edge_tts_trimmed_generation_time_sec"
]].describe()

,edge_tts_generation_time_sec,edge_tts_trimmed_raw_generation_time_sec,edge_tts_trimmed_trim_time_sec,edge_tts_trimmed_generation_time_sec
count,786.000000,786.000000,786.000000,786.000000
mean,1.020984,0.677942,0.021583,0.699525
std,0.637608,0.330410,0.005893,0.330627
min,0.533668,0.508006,0.009049,0.519783
25%,0.673116,0.618182,0.018550,0.639087
50%,0.909299,0.653375,0.021290,0.674677
75%,1.224553,0.689735,0.023964,0.712878
max,11.128465,8.104157,0.121893,8.124345


In [59]:
def summarize_entity_metrics(sent_df):
    entity_name = sent_df["entity_name"].iloc[0]

    pocket_word_total = sent_df["pocket_tts_word_count"].sum()
    edge_word_total = sent_df["edge_tts_word_count"].sum()
    edge_trimmed_word_total = sent_df["edge_tts_trimmed_word_count"].sum()

    pocket_duration_total = sent_df["pocket_tts_duration_sec"].sum()
    edge_duration_total = sent_df["edge_tts_duration_sec"].sum()
    edge_trimmed_duration_total = sent_df["edge_tts_trimmed_duration_sec"].sum()

    return pd.DataFrame([{
        "entity_name": entity_name,

        # Pocket-TTS
        "pocket_tts_wpm": (pocket_word_total / pocket_duration_total) * 60 if pocket_duration_total > 0 else np.nan,
        "pocket_tts_wer": (
            (sent_df["pocket_tts_wer"] * sent_df["pocket_tts_word_count"]).sum() / pocket_word_total
            if pocket_word_total > 0 else np.nan
        ),
        "pocket_tts_silence_ratio": sent_df["pocket_tts_silence_ratio"].mean(),
        "pocket_tts_generation_time_sec": sent_df["pocket_tts_generation_time_sec"].sum(),

        # Edge-TTS
        "edge_tts_wpm": (edge_word_total / edge_duration_total) * 60 if edge_duration_total > 0 else np.nan,
        "edge_tts_wer": (
            (sent_df["edge_tts_wer"] * sent_df["edge_tts_word_count"]).sum() / edge_word_total
            if edge_word_total > 0 else np.nan
        ),
        "edge_tts_silence_ratio": sent_df["edge_tts_silence_ratio"].mean(),
        "edge_tts_generation_time_sec": sent_df["edge_tts_generation_time_sec"].sum(),
    
        # Edge-TTS Trimmed
        "edge_tts_trimmed_wpm": (edge_trimmed_word_total / edge_trimmed_duration_total) * 60 if edge_trimmed_duration_total > 0 else np.nan,
        "edge_tts_trimmed_wer": (
            (sent_df["edge_tts_trimmed_wer"] * sent_df["edge_tts_trimmed_word_count"]).sum() / edge_trimmed_word_total
            if edge_trimmed_word_total > 0 else np.nan
        ),
        "edge_tts_trimmed_silence_ratio": sent_df["edge_tts_trimmed_silence_ratio"].mean(),
        "edge_tts_trimmed_generation_time_sec": sent_df["edge_tts_trimmed_generation_time_sec"].mean(),

        "edge_tts_leading_silence_sec": sent_df["edge_tts_leading_silence_sec"].mean(),
        "edge_tts_trailing_silence_sec": sent_df["edge_tts_trailing_silence_sec"].mean(),
        "edge_tts_total_outer_silence_sec": sent_df["edge_tts_total_outer_silence_sec"].mean(),
        "edge_tts_outer_silence_ratio": sent_df["edge_tts_outer_silence_ratio"].mean(),
    }])

In [65]:
all_sent_df_updated.to_csv(
    "Outputs/audio_eval/final_sentence_results_with_trimmed.csv",
    index=False
)

entity_summary_df_updated = (
    all_sent_df_updated.groupby("entity_name", group_keys=False)
    .apply(summarize_entity_metrics)
    .reset_index(drop=True)
)

entity_summary_df_updated.to_csv(
    "Outputs/audio_eval/final_entity_summary_with_trimmed.csv",
    index=False
)

C:\Users\Malak Diab\AppData\Local\Temp\ipykernel_24148\2900981339.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_entity_metrics)


In [64]:
final_avg_across_entities = pd.DataFrame({
    "model": ["pocket_tts", "edge_tts", "edge_tts_trimmed"],

    "avg_wpm_across_80_entities": [
        entity_summary_df_updated["pocket_tts_wpm"].mean(),
        entity_summary_df_updated["edge_tts_wpm"].mean(),
        entity_summary_df_updated["edge_tts_trimmed_wpm"].mean()
    ],

    "avg_wer_across_80_entities": [
        entity_summary_df_updated["pocket_tts_wer"].mean(),
        entity_summary_df_updated["edge_tts_wer"].mean(),
        entity_summary_df_updated["edge_tts_trimmed_wer"].mean()
    ],

    "avg_silence_ratio_across_80_entities": [
        entity_summary_df_updated["pocket_tts_silence_ratio"].mean(),
        entity_summary_df_updated["edge_tts_silence_ratio"].mean(),
        entity_summary_df_updated["edge_tts_trimmed_silence_ratio"].mean()
    ],

    "avg_generation_time_sec_across_80_entities": [
        entity_summary_df_updated["pocket_tts_generation_time_sec"].mean(),
        entity_summary_df_updated["edge_tts_generation_time_sec"].mean(),
        entity_summary_df_updated["edge_tts_trimmed_generation_time_sec"].mean()
    ]
})

final_avg_across_entities

,model,avg_wpm_across_80_entities,avg_wer_across_80_entities,avg_silence_ratio_across_80_entities,avg_generation_time_sec_across_80_entities
0,pocket_tts,160.268440,0.097583,0.079502,29.283830
1,edge_tts,146.866449,0.099106,0.183899,10.031166
2,edge_tts_trimmed,170.028598,0.098785,0.051306,0.699034


### Audio

In [10]:
#Text-to-Speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

i = 0
for p in sentences:
    for s in p:
        audio = tts_model.generate_audio(voice_state, s)
        scipy.io.wavfile.write(f"Outputs\\output_{i}.wav", tts_model.sample_rate, audio.numpy())
        i += 1

In [11]:
wav_files = sorted([f for f in os.listdir("Outputs") if f.endswith('.wav')])
print(wav_files)

audio_data = []
seconds_per_paragraph = []
samplerate = None

for i in range(len(wav_files)):
    file_name = f"Outputs\\output_{i}.wav"
    
    data, sr = sf.read(file_name)
    
    if samplerate is None:
        samplerate = sr
    elif sr != samplerate:
        raise ValueError("Sample rates do not match!")

    audio_data.append(data)

# Concatenate along time axis
combined = np.concatenate(audio_data, axis=0)

# Write output (keeps float format safely)
sf.write("Outputs\\Final audio.wav", combined, samplerate)

['output_0.wav', 'output_1.wav', 'output_10.wav', 'output_11.wav', 'output_2.wav', 'output_3.wav', 'output_4.wav', 'output_5.wav', 'output_6.wav', 'output_7.wav', 'output_8.wav', 'output_9.wav']


In [12]:
durations = []
sentences_durations = []
images_needed = []
#seconds = []
i = 0
for p in sentences:
    duration_seconds = 0
    for s in p:
        file_path = f"Outputs\\output_{i}.wav"
        # Returns the sample rate (Fs) and the data as a NumPy array
        Fs, data = wavfile.read(file_path) 
        # The length of the data array divided by the sample rate gives the duration
        sentence_duration = len(data) / float(Fs)
        duration_seconds += sentence_duration
        sentences_durations.append(sentence_duration)
        i += 1
        os.remove(file_path)
    durations.append(duration_seconds)
    images_needed.append(max(1, int(duration_seconds / 6)))
    '''    section_seconds = duration_seconds / images_needed[-1]
    for img in range(images_needed[-1]):
        seconds.append(section_seconds)'''


print(f"Durations of audio files: {durations}")
print(f"Images needed for each paragraph: {images_needed}")
print(f"Durations of sentences: {sentences_durations}")
#print(f"Seconds for each image: {seconds}")

Durations of audio files: [25.92, 20.48, 15.2, 21.28]
Images needed for each paragraph: [4, 3, 2, 3]
Durations of sentences: [4.32, 8.56, 8.4, 4.64, 5.84, 7.76, 6.88, 7.44, 7.76, 10.8, 5.44, 5.04]


In [13]:
i = 0
sentence_start = 0
sentence_end = 0
image_text_chunks = []
seconds_for_chunk = []

# Use the already-split sentences from the first pass (list of lists)
for para_idx, para_sentences in enumerate(sentences_per_paragraph):
    # sentences_split is your original `sentences` list of lists
    para_sentence_list = sentences[para_idx]
    images_for_paragraph = images_needed[para_idx]
    images_for_paragraph = min(images_for_paragraph, len(para_sentence_list))

    total_sentences = len(para_sentence_list)
    base = total_sentences // images_for_paragraph
    remainder = total_sentences % images_for_paragraph

    groups = []
    start = 0
    for img_idx in range(images_for_paragraph):
        extra = 1 if img_idx < remainder else 0
        end = start + base + extra

        chunk_sentence_end = sentence_start + end  # absolute index into sentences_durations
        chunk_duration = sum(sentences_durations[sentence_start + start : sentence_start + end])
        seconds_for_chunk.append(chunk_duration)

        chunk = " ".join(para_sentence_list[start:end])
        groups.append(chunk)

        start = end

    sentence_start += total_sentences  # advance global pointer by paragraph's sentence count
    image_text_chunks.append(groups)

print(f"Text chunks for images: {image_text_chunks}")
print(f"Seconds for each chunk: {seconds_for_chunk}")

Text chunks for images: [['Rameses II was one of ancient Egypt’s greatest pharaohs.', 'Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.', 'At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.', 'He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.'], ['Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.', 'This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.', 'Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.'], ["He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength.", 'Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first 

### Retrieval Approaches

In [14]:
def cosine(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

In [47]:
import ast
def retrieve_images_for_chunks(NAME):
    fetched_images_ids = []
    fetched_images_paths = []
    repeated_images_ids = []
    scores = []
    trials = []
    for paragraph_chunks in image_text_chunks:
        for chunk in paragraph_chunks:
            text_tokens = open_clip.get_tokenizer()([chunk]).to(device)
        
            with torch.no_grad():
                text_embedding = model.encode_text(text_tokens)
                text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

            embedding_list = text_embedding.cpu().numpy().tolist()[0]
            if is_landmark:
                query = "SELECT li.id, li.image_path, li.image_embedding::text FROM landmark_images as li JOIN landmarks as l ON li.landmark_id = l.id WHERE l.name = :name ORDER BY image_embedding <=> CAST(:embedding AS vector)"
            else:
                query = "SELECT pi.id, pi.image_path, pi.image_embedding::text FROM pharaohs_images as pi JOIN pharaohs as p ON pi.pharaoh_id = p.id where p.name = :name ORDER BY image_embedding <=> CAST(:embedding AS vector)"


            with Session(engine) as session:
                result = session.execute(text(query), {"name": NAME, "embedding": embedding_list})
                images = result.fetchall()
                image_id = images[0][0]
                image_path = images[0][1]
                image_embedding = images[0][2]
                image_embedding = np.fromstring(image_embedding.strip("[]"), sep=",", dtype=np.float32)
                j = 0
                while image_id in fetched_images_ids:
                    images.pop(0)
                    if not images:
                        #print("No more unique images available for this chunk. Duplicate images will be used.")
                        result = session.execute(text(query), {"name": NAME, "embedding": embedding_list})
                        images = result.fetchall()
                        image_id = images[0][0]
                        while image_id in repeated_images_ids:
                            images.pop(0)
                            image_id = images[0][0]
                            image_path = images[0][1]
                            image_embedding = images[0][2]
                            image_embedding = np.fromstring(image_embedding.strip("[]"), sep=",", dtype=np.float32)
                            j += 1
                        repeated_images_ids.append(image_id)
                        break
                    image_id = images[0][0]
                    image_path = images[0][1]
                    image_embedding = images[0][2]
                    image_embedding = np.fromstring(image_embedding.strip("[]"), sep=",", dtype=np.float32)
                    j += 1
                score = cosine(text_embedding.cpu().numpy()[0], image_embedding)
                scores.append(score)
                fetched_images_ids.append(image_id)
                fetched_images_paths.append(image_path)
                trials.append(j)
            
        
            #print(f"Text chunk: {chunk}")
            #print(f"Image ID: {image_id} on trial {j}")
            #print(f"image path: {image_path}")
    average_score = sum(scores) / len(scores) if scores else 0
    average_trials = sum(trials) / len(trials) if trials else 0
    return fetched_images_ids, fetched_images_paths, average_score, average_trials

In [15]:
fetched_images_ids, fetched_images_paths, score, trial = retrieve_images_for_chunks()

In [15]:
IMAGE_SIM_WEIGHT = 0.3
DESC_SIM_WEIGHT = 0.7

def cosine(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

In [49]:
def retrieve_images_semantic(NAME):
    fetched_images_ids = []
    fetched_images_paths = []
    repeated_images_ids = []
    best_scores = []
    trials = []
    with Session(engine) as session:
        result = session.execute(
            text("""
                SELECT 
                    pi.id,
                    pi.image_path,
                    pi.image_embedding,
                    pi.image_description
                FROM pharaohs_images pi
                JOIN pharaohs p ON pi.pharaoh_id = p.id
                WHERE p.name = :name
            """),
            {"name": NAME}
        )

        images_data=result.fetchall()
        
    clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
    processed_images=[]
    for row in images_data:
        image_id,image_path,image_embedding,image_description=row
        if isinstance(image_embedding,str):
            image_embedding=json.loads(image_embedding)

        image_embedding=np.array(image_embedding)
        image_embedding=image_embedding/np.linalg.norm(image_embedding)

        tokens=clip_tokenizer([image_description]).to(device)

        with torch.no_grad():
            desc_emb=model.encode_text(tokens)
            desc_emb/=desc_emb.norm(dim=-1,keepdim=True)

        desc_emb=desc_emb.cpu().numpy()[0]

        processed_images.append({
            "id": image_id,
            "path": image_path,
            "img_emb": image_embedding,
            "desc_emb": desc_emb
        })
    
    for paragraph_chunks in image_text_chunks:
        for chunk in paragraph_chunks:
            text_tokens = open_clip.get_tokenizer()([chunk]).to(device)
            with torch.no_grad():
                emb = model.encode_text(text_tokens)
                emb /= emb.norm(dim=-1, keepdim=True)
            scene_emb = emb.cpu().numpy()[0]

            ranked = []
            for img in processed_images:
                image_sim = cosine(scene_emb, img["img_emb"])
                desc_sim = cosine(scene_emb, img["desc_emb"])
                score = IMAGE_SIM_WEIGHT * image_sim + DESC_SIM_WEIGHT * desc_sim
                ranked.append((score, img["id"], img["path"]))
                
            ranked.sort(reverse=True, key=lambda x: x[0])
            ranked2= ranked.copy()
            best_score, best_id, best_path = ranked[0]
            j = 0
            while best_id in fetched_images_ids:
                ranked.pop(0)
                j += 1
                if not ranked:
                    #print("No more unique images available for this chunk. Duplicate images will be used.")
                    best_score, best_id, best_path = ranked2[0]
                    while best_id in repeated_images_ids:
                        ranked2.pop(0)
                        j += 1
                        if not ranked2:
                            print("No more unique images available at all.")
                            break
                        best_score, best_id, best_path = ranked2[0]
                    break
                best_score, best_id, best_path = ranked[0]
            fetched_images_ids.append(best_id)
            fetched_images_paths.append(best_path)
            best_scores.append(best_score)
            trials.append(j)
            #print(f"Text chunk: {chunk}")
            #print(f"Best image ID: {best_id} with score {best_score}")
            #print(f"Best image path: {best_path}")
    average_score = sum(best_scores) / len(best_scores) if best_scores else 0
    average_trials = sum(trials) / len(trials) if trials else 0
    return fetched_images_ids, fetched_images_paths, average_score, average_trials

In [19]:
fetched_images_ids, fetched_images_paths, score, trials = retrieve_images_semantic()

In [55]:
import time
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

def generate_visual_prompt(scene_text):

    prompt = f"""
You are an expert visual prompt generator for an Ancient Egypt image retrieval system.

Convert the narration into a SHORT visual description of what an image should show, that will help retrieve the best matching historical image from an Ancient Egypt dataset.


The image database contains:
- Pharaoh statues
- Egyptian gods and goddesses
- Royal family members (father, son, wife, daughter)
- Temple relief carvings
- Archaeological artifacts
- Monuments and temples

Guidelines:
1. Identify the MAIN ENTITY (pharaoh, god, goddess, royal family).
2. Include the NAME if present.
3. Include VISUAL ELEMENTS such as: statue, temple relief, wall carving, hieroglyphs, monument, artifact.
4. Include LOCATION if mentioned (temple name, city, monument).
5. Prefer descriptions that look like museum or archaeological images.
6. Use concise keyword-style phrases, not storytelling.

Output a single visual description.

Narration:
{scene_text}

Visual prompt:
"""

    time.sleep(0.5)
    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
            {"role": "system", "content": "You create visual prompts for documentary scenes."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

def retrieve_images_image_oriental(NAME):
    fetched_images_ids = []
    fetched_images_paths = []
    repeated_images_ids = []
    best_scores = []
    trials = []
    with Session(engine) as session:
        result = session.execute(
            text("""
                SELECT 
                    pi.id,
                    pi.image_path,
                    pi.image_embedding,
                    pi.image_description
                FROM pharaohs_images pi
                JOIN pharaohs p ON pi.pharaoh_id = p.id
                WHERE p.name = :name
            """),
            {"name": NAME}
        )

        images_data=result.fetchall()

    clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
    processed_images=[]
    for row in images_data:
        image_id,image_path,image_embedding,image_description=row
        if isinstance(image_embedding,str):
            image_embedding=json.loads(image_embedding)

        image_embedding=np.array(image_embedding)
        image_embedding=image_embedding/np.linalg.norm(image_embedding)

        tokens=clip_tokenizer([image_description]).to(device)

        with torch.no_grad():
            desc_emb=model.encode_text(tokens)
            desc_emb/=desc_emb.norm(dim=-1,keepdim=True)

        desc_emb=desc_emb.cpu().numpy()[0]

        processed_images.append({
            "id": image_id,
            "path": image_path,
            "img_emb": image_embedding,
            "desc_emb": desc_emb
        })
    for paragraph_chunks in image_text_chunks:
        for chunk in paragraph_chunks:
            visual_prompt = generate_visual_prompt(chunk)
            prompt_tokens = clip_tokenizer([visual_prompt]).to(device)
            with torch.no_grad():
                prompt_emb = model.encode_text(prompt_tokens)
                prompt_emb /= prompt_emb.norm(dim=-1, keepdim=True)
            prompt_emb = prompt_emb.cpu().numpy()[0]

            text_tokens = clip_tokenizer([chunk]).to(device)
            with torch.no_grad():
                text_emb = model.encode_text(text_tokens)
                text_emb /= text_emb.norm(dim=-1, keepdim=True)
            text_emb = text_emb.cpu().numpy()[0]

            ranked = []
            for img in processed_images:
                image_sim = cosine(prompt_emb, img["img_emb"])
                desc_sim = cosine(text_emb, img["desc_emb"])
                score = IMAGE_SIM_WEIGHT * image_sim + DESC_SIM_WEIGHT * desc_sim
                ranked.append((score, img["id"], img["path"]))
            ranked.sort(reverse=True, key=lambda x: x[0])

            ranked2= ranked.copy()
            best_score, best_id, best_path = ranked[0]
            j = 0
            while best_id in fetched_images_ids:
                ranked.pop(0)
                j += 1
                if not ranked:
                    #print("No more unique images available for this chunk. Duplicate images will be used.")
                    best_score, best_id, best_path = ranked2[0]
                    while best_id in repeated_images_ids:
                        ranked2.pop(0)
                        j += 1
                        if not ranked2:
                            print("No more unique images available at all.")
                            break
                        best_score, best_id, best_path = ranked2[0]
                    break
                best_score, best_id, best_path = ranked[0]
            fetched_images_ids.append(best_id)
            fetched_images_paths.append(best_path)
            best_scores.append(best_score)
            trials.append(j)
            #print(f"Text chunk: {chunk}")
            #print(f"Best image ID: {best_id} with score {best_score}")
            #print(f"Best image path: {best_path}")
    average_score = sum(best_scores) / len(best_scores) if best_scores else 0
    average_trials = sum(trials) / len(trials) if trials else 0
    return fetched_images_ids, fetched_images_paths, average_score, average_trials

In [ ]:
fetched_images_ids, fetched_images_paths, best_scores = retrieve_images_image_oriental()

### Retrieval Evaluation

In [43]:
with Session(engine) as session:
        result = session.execute(text("select p.name from pharaohs_scripts as ps, pharaohs as p where p.id = ps.id"))
        names = [row[0] for row in result.fetchall()]
names.remove('Amenhotep III and God Re-Horakhty')
names.remove('Amenhotep III and Wife Queen Tiye')
names.remove('Amun and Mut (God & Goddess)')
names.remove('Hor (son of Ankh-Khonsu)')
names.remove('Itysen (Prince, probably son of Djedefre)')
names.remove('Isis (Goddess) with her child')
names.remove('Menkaure and his wife (likely Queen Khamerernebty II)')
names.remove('Menkaure with Goddesses Hathor and Bat')
names.remove('Merenptah and Goddess Mut')
names.remove('Ramesses II and God Ptah and Goddess Sekhment')
names.remove('Nectanebo II and God Horus')
names.remove('Osirian Triad and a King (Osiris, Isis, Horus, and Hormheb)')
names.remove('Rameses II and Hauron')
names.remove('Ramesess II and God Amun & Goddess Mut')
names.remove('Ramesses II and God Ptah')
names.remove('Ramesses II and God Ptah and a Goddess')
names.remove('Ramesses II and Goddess')
names.remove('Ramesses II and Goddess Anath')
names.remove('Ramesses II and Goddesses Isis and Hathor')
names.remove('Ramesses III and Gods Horus and Seth')
names.remove('Hauron God')
names

['Achoris',
 'Ahmose I',
 'Akhenaton',
 'Alexander The Great',
 'Amasis II',
 'Amenemhat I',
 'Amenemhat III',
 'Amenhotep II',
 'Amenhotep III',
 'Amenirdis (Daughter of Kashta)',
 'Amun (God)',
 'Amun-Ra (God)',
 'Anath Goddess',
 'Ankhsenamun',
 'Anubis (God)',
 'Arsinoe II (Queen, sister and wife of Ptolemy IV)',
 'Arsinoe III (Queen, daughter of Ptolemy I and wife of Ptolemy II)',
 'Bat Goddess',
 'Cleopatra VII Philopator',
 'Djoser',
 'Hathor (Goddess)',
 'Hatshepsut',
 'Hor I',
 'Horemheb',
 'Horus (God)',
 'Isis (Goddess)',
 'Isis (mother of Thutmose III)',
 'Khamerernebty II',
 'Khafre',
 'Khasekhem',
 'Khonsu (God)',
 'Khufu',
 'Menkuare',
 'Mentuhotep II',
 'Merenptah',
 'Meresankh (Queen, wife of Khafre)',
 'Mut (Goddess)',
 'Mutnofret (Queen, Wife of Thutmose I and mother of Thutmose II)',
 'Nectanebo I',
 'Nectanebo II',
 'Nefertiti',
 'Nofret (Queen, possibly Nofret II, wife of Senusret II)',
 'Niuserre',
 'Osiris (God)',
 'Osorkon II',
 'Pepi I',
 'Psusennes I',
 'Ptah

In [48]:
scores = []
trials = []
for name in names:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": name})
        script = result.fetchall()[0][0]

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
    sentences = []
    sentences_per_paragraph = []
    for p in paragraphs:
        sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
        sentences_per_paragraph.append(len(sentences[-1]))
        
    best_images_ids, best_images_paths, average_score, average_trials = retrieve_images_for_chunks(name)
    scores.append(average_score)
    trials.append(average_trials)
    print(f"Pharaoh: {name}, Average CLIP score: {scores[-1]}, Average retrieval trials: {trials[-1]}")
print(f"Scores for each pharaoh: {dict(zip(names, scores))}")
print(f"Overall average score for chunks approach: {sum(scores) / len(scores) if scores else 0}")
print(f"Overall average retrieval trials for chunks approach: {sum(trials) / len(trials) if trials else 0}")

Pharaoh: Achoris, Average CLIP score: 0.23382891714572906, Average retrieval trials: 3.4166666666666665
Pharaoh: Ahmose I, Average CLIP score: 0.25675880908966064, Average retrieval trials: 2.4166666666666665
Pharaoh: Akhenaton, Average CLIP score: 0.2742162048816681, Average retrieval trials: 1.4166666666666667
Pharaoh: Alexander The Great, Average CLIP score: 0.22664785385131836, Average retrieval trials: 3.75
Pharaoh: Amasis II, Average CLIP score: 0.2341011017560959, Average retrieval trials: 3.25
Pharaoh: Amenemhat I, Average CLIP score: 0.26321345567703247, Average retrieval trials: 2.25
Pharaoh: Amenemhat III, Average CLIP score: 0.26488155126571655, Average retrieval trials: 2.3333333333333335
Pharaoh: Amenhotep II, Average CLIP score: 0.251228004693985, Average retrieval trials: 2.9166666666666665
Pharaoh: Amenhotep III, Average CLIP score: 0.2769683003425598, Average retrieval trials: 1.5
Pharaoh: Amenirdis (Daughter of Kashta), Average CLIP score: 0.2349942922592163, Average

In [50]:
scores = []
trials = []
for name in names:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": name})
        script = result.fetchall()[0][0]

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
    sentences = []
    sentences_per_paragraph = []
    for p in paragraphs:
        sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
        sentences_per_paragraph.append(len(sentences[-1]))
        
    best_images_ids, best_images_paths, average_score, average_trials = retrieve_images_semantic(name)
    scores.append(average_score)
    trials.append(average_trials)
    print(f"Pharaoh: {name}, Average CLIP score: {scores[-1]}, Average retrieval trials: {trials[-1]}")
print(f"Averages for each pharaoh: {dict(zip(names, scores))}")
print(f"Overall average score for semantic approach: {sum(scores) / len(scores) if scores else 0}")
print(f"Overall average retrieval trials for semantic approach: {sum(trials) / len(trials) if trials else 0}")

Pharaoh: Achoris, Average CLIP score: 0.3322843442467156, Average retrieval trials: 4.333333333333333
Pharaoh: Ahmose I, Average CLIP score: 0.40120287670620897, Average retrieval trials: 1.75
Pharaoh: Akhenaton, Average CLIP score: 0.3615270674593907, Average retrieval trials: 3.0833333333333335
Pharaoh: Alexander The Great, Average CLIP score: 0.3427146516550097, Average retrieval trials: 2.5833333333333335
Pharaoh: Amasis II, Average CLIP score: 0.3591540892910972, Average retrieval trials: 3.9166666666666665
Pharaoh: Amenemhat I, Average CLIP score: 0.38436145533855476, Average retrieval trials: 2.3333333333333335
Pharaoh: Amenemhat III, Average CLIP score: 0.3683940179388739, Average retrieval trials: 2.0833333333333335
Pharaoh: Amenhotep II, Average CLIP score: 0.38143150365721795, Average retrieval trials: 2.9166666666666665
Pharaoh: Amenhotep III, Average CLIP score: 0.4007703776625029, Average retrieval trials: 2.0833333333333335
Pharaoh: Amenirdis (Daughter of Kashta), Averag

In [56]:
scores = []
trials = []
for name in names:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": name})
        script = result.fetchall()[0][0]

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
    sentences = []
    sentences_per_paragraph = []
    for p in paragraphs:
        sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
        sentences_per_paragraph.append(len(sentences[-1]))
        
    best_images_ids, best_images_paths, average_score, average_trials = retrieve_images_image_oriental(name)
    scores.append(average_score)
    trials.append(average_trials)
    print(f"Pharaoh: {name}, Average CLIP score: {scores[-1]}")
print(f"Averages for each pharaoh: {dict(zip(names, scores))}")
print(f"Overall average score for image-oriented approach: {sum(scores) / len(scores) if scores else 0}")
print(f"Overall average retrieval trials for image-oriented approach: {sum(trials) / len(trials) if trials else 0}")

Pharaoh: Achoris, Average CLIP score: 0.32785840634381186


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kjwsyj6ye8stmfq6c6b93bdn` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5148, Requested 929. Please try again in 770ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Loading images from Cloudflare

In [20]:
load_dotenv()

ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
SECRET_KEY = os.getenv("R2_SECRET_KEY")
BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

session = boto3.session.Session()
client = session.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)
if is_landmark:
    path = Path.joinpath(Path("data/video_generation/raw/landmark_images/"), NAME.replace(" ", "_").lower())
else:
    path = Path.joinpath(Path("data/video_generation/raw/pharaohs_images/"), NAME.replace(" ", "_").lower())
REMOTE_PREFIX = path

local_frames_dir = Path("temp_frames")
local_frames_dir.mkdir(exist_ok=True)

ordered_local_paths = []

def download_image(idx_key):
    idx, image_key = idx_key
    local_file = local_frames_dir / f"{idx:04d}.jpg"
    client.download_file(BUCKET_NAME, image_key, str(local_file))
    return str(local_file)

with ThreadPoolExecutor(max_workers=8) as executor:
    ordered_local_paths = list(
        executor.map(download_image, enumerate(fetched_images_paths))
    )

In [21]:
image_files = list(local_frames_dir.glob("*.jpg")) + list(local_frames_dir.glob("*.jpeg"))
image_files

[WindowsPath('temp_frames/0000.jpg'),
 WindowsPath('temp_frames/0001.jpg'),
 WindowsPath('temp_frames/0002.jpg'),
 WindowsPath('temp_frames/0003.jpg'),
 WindowsPath('temp_frames/0004.jpg'),
 WindowsPath('temp_frames/0005.jpg'),
 WindowsPath('temp_frames/0006.jpg'),
 WindowsPath('temp_frames/0007.jpg'),
 WindowsPath('temp_frames/0008.jpg'),
 WindowsPath('temp_frames/0009.jpg'),
 WindowsPath('temp_frames/0010.jpg'),
 WindowsPath('temp_frames/0011.jpg')]

In [22]:
from PIL import Image as PILImage
for path in image_files:
    p = Path(path)
    try:
        with PILImage.open(p) as img:
            img = img.convert("RGB")  # handles AVIF, PNG, RGBA, etc.
            img.save(p, "JPEG", quality=95)
        print(f"Converted: {p}")
    except Exception as e:
        print(f"Failed to convert {p}: {e}")

Converted: temp_frames\0000.jpg
Converted: temp_frames\0001.jpg
Converted: temp_frames\0002.jpg
Converted: temp_frames\0003.jpg
Converted: temp_frames\0004.jpg
Converted: temp_frames\0005.jpg
Converted: temp_frames\0006.jpg
Converted: temp_frames\0007.jpg
Converted: temp_frames\0008.jpg
Converted: temp_frames\0009.jpg
Converted: temp_frames\0010.jpg
Converted: temp_frames\0011.jpg


In [23]:
for img_path in Path("temp_frames").glob("*.jpg"):
    print(img_path, img_path.stat().st_size)
    try:
        with Image.open(img_path) as im:
            print("OK:", im.format, im.size)
    except Exception as e:
        print("Broken image:", e)

temp_frames\0000.jpg 113340
OK: JPEG (529, 395)
temp_frames\0001.jpg 315849
OK: JPEG (849, 1390)
temp_frames\0002.jpg 97465
OK: JPEG (522, 345)
temp_frames\0003.jpg 861547
OK: JPEG (1400, 1080)
temp_frames\0004.jpg 163083
OK: JPEG (559, 559)
temp_frames\0005.jpg 273076
OK: JPEG (1200, 675)
temp_frames\0006.jpg 7435932
OK: JPEG (4284, 5712)
temp_frames\0007.jpg 98702
OK: JPEG (594, 391)
temp_frames\0008.jpg 218494
OK: JPEG (875, 577)
temp_frames\0009.jpg 78597
OK: JPEG (546, 362)
temp_frames\0010.jpg 1317111
OK: JPEG (2500, 1667)
temp_frames\0011.jpg 222794
OK: JPEG (798, 590)


### Subtitles

In [24]:
# ---------- SETTINGS ----------
MAX_CHARS_PER_LINE = 35
MAX_LINES = 2
MIN_DURATION = 1.0  # minimum subtitle duration in seconds

# ---------- HELPERS ----------
def normalize_text(text):
    """
    Cleans problematic Unicode characters that break
    subtitle rendering in Pillow / MoviePy.
    """

    # First normalize Unicode composition
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",   # non-breaking space
        "\u200B": "",    # zero-width space
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",    # BOM if embedded
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove any remaining control characters
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text

def format_timestamp(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    # combine into blocks of max 2 lines
    blocks = []
    for i in range(0, len(lines), MAX_LINES):
        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks

'''
# ---------- MAIN FUNCTION ----------
def generate_srt(paragraphs, durations, output_path="subtitles.srt"):
    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for paragraph, duration in zip(paragraphs, durations):
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        # break into subtitle chunks
        chunks = []
        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        for chunk in chunks:
            chunk_char_count = len(chunk.replace("\n", ""))

            # proportional timing
            chunk_duration = max(
                MIN_DURATION,
                (chunk_char_count / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
            srt_blocks.append(srt_block)

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, durations, "Outputs/output_subtitles.srt")'''

'\n# ---------- MAIN FUNCTION ----------\ndef generate_srt(paragraphs, durations, output_path="subtitles.srt"):\n    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"\n\n    current_time = 0.0\n    subtitle_index = 1\n    srt_blocks = []\n\n    for paragraph, duration in zip(paragraphs, durations):\n        paragraph = normalize_text(paragraph)\n        sentences = split_into_sentences(paragraph)\n\n        # break into subtitle chunks\n        chunks = []\n        for sentence in sentences:\n            chunks.extend(split_long_text(sentence))\n\n        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)\n\n        for chunk in chunks:\n            chunk_char_count = len(chunk.replace("\n", ""))\n\n            # proportional timing\n            chunk_duration = max(\n                MIN_DURATION,\n                (chunk_char_count / total_chars) * duration\n            )\n\n            start_time = current_time\n            end_time = cur

In [25]:
def generate_srt(paragraphs, sentences_durations, output_path="subtitles.srt"):
    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []
    duration_index = 0  # pointer into flat sentences_durations list

    for paragraph in paragraphs:
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        for sentence in sentences:
            sentence_duration = sentences_durations[duration_index]
            duration_index += 1

            # Split long sentence into visual chunks
            chunks = split_long_text(sentence)
            total_chars = sum(len(c.replace("\n", "")) for c in chunks)

            for chunk in chunks:
                chunk_char_count = len(chunk.replace("\n", ""))

                # Proportional split only within the sentence now
                chunk_duration = max(
                    MIN_DURATION,
                    (chunk_char_count / total_chars) * sentence_duration
                )

                start_time = current_time
                end_time = current_time + chunk_duration

                srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
                srt_blocks.append(srt_block)
                current_time = end_time
                subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, sentences_durations, "Outputs/output_subtitles.srt")

SRT file saved to Outputs/output_subtitles.srt


### Main Pipeline

In [26]:
def run_ffmpeg(cmd):
    cmd = [str(c) for c in cmd]
    print("Running:", " ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stderr)  # show FFmpeg error

    if result.returncode != 0:
        raise RuntimeError("FFmpeg failed")

FFMPEG  = shutil.which("ffmpeg")

In [27]:
def _stable_int(seed: str) -> int:
    return int(hashlib.md5(seed.encode("utf-8")).hexdigest()[:8], 16)

def _stable_unit(seed: str) -> float:
    return (_stable_int(seed) % 10_000) / 10_000.0

In [28]:
def plan_kenburns_sequence(
    image_files,
    durations,
    target_w=1920,
    target_h=1080,
    threshold=40,
    min_pan_travel=140,
    max_pan_speed=120,
    vertical_travel_ratio=0.80,
    horizontal_travel_ratio=0.55,
):
    PALETTE = [
        ("zoom", "in"),
        ("pan",  "forward"),
        ("zoom", "out"),
        ("pan",  "reverse"),
    ]

    plans       = []
    palette_idx = 0
    last_sig    = None
    target_ar   = target_w / target_h

    for img_path, duration in zip(image_files, durations):
        img_path = Path(img_path)

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        scale_factor = max(target_w / img_w, target_h / img_h)
        scaled_w     = int(round(img_w * scale_factor))
        scaled_h     = int(round(img_h * scale_factor))

        scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
        scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1
        
        max_x        = max(0, scaled_w - target_w)
        max_y        = max(0, scaled_h - target_h)

        img_ar      = img_w / img_h
        is_vertical = img_ar < target_ar
        pan_axis    = "vertical" if is_vertical else "horizontal"

        travel_x = min(max_x, min(int(max_pan_speed * duration),
                                  int(max_x * horizontal_travel_ratio)))
        travel_y = min(max_y, min(int(max_pan_speed * duration),
                                  int(max_y * vertical_travel_ratio)))
        can_pan  = (travel_y > max(threshold, min_pan_travel)) if is_vertical \
                   else (travel_x > max(threshold, min_pan_travel))

        mode = direction = None
        for attempt in range(len(PALETTE)):
            cand_mode, cand_dir = PALETTE[(palette_idx + attempt) % len(PALETTE)]
            if cand_mode == "pan" and not can_pan:
                continue
            if (cand_mode, cand_dir) == last_sig:
                continue
            mode, direction = cand_mode, cand_dir
            palette_idx = (palette_idx + attempt + 1) % len(PALETTE)
            break

        if mode is None:
            mode        = "zoom"
            direction   = "out" if (last_sig and last_sig[1] == "in") else "in"
            palette_idx = (palette_idx + 1) % len(PALETTE)

        last_sig = (mode, direction)
        plans.append({
            "mode":        mode,
            "direction":   direction,
            "is_vertical": is_vertical,
            "pan_axis":    pan_axis,
        })

    return plans


In [29]:
def create_kenburns_clip(
    image_path,
    duration,
    output_path,
    fps=30,
    target_w=1920,
    target_h=1080,

    threshold=40,
    min_pan_travel=140,
    prefer_pan_when_possible=True,

    zoom_min=1.03,
    zoom_max=1.08,
    zoom_rate_per_sec=0.055,
    zoom_anchor_vertical_y=0.18,
    zoom_anchor_horizontal_y=0.50,

    max_pan_speed=200,
    vertical_travel_ratio=0.80,
    horizontal_travel_ratio=0.55,
    top_bias=0.00,

    motion_scale=1.4,
    use_nvenc=True,

    motion_mode=None,
    motion_direction=None,
):
    total_frames = max(2, int(round(duration * fps)))
    image_path   = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    scale_factor = max(target_w / img_w, target_h / img_h)
    scaled_w = (int(round(img_w * scale_factor)) )
    scaled_h = (int(round(img_h * scale_factor)) ) 

    scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
    scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1
        
    max_x = max(0, scaled_w - target_w)
    max_y = max(0, scaled_h - target_h)

    denom    = total_frames - 1
    ease_pan  = f"(0.5-0.5*cos(PI*n/{denom}))"
    ease_zoom = f"(0.5-0.5*cos(PI*on/{denom}))"

    target_ar   = target_w / target_h
    img_ar      = img_w / img_h
    is_vertical = img_ar < target_ar

    u = _stable_unit(str(image_path))

    zoom_direction = "out" if u < 0.5 else "in"
    pan_reverse    = u > 0.66

    if motion_direction is not None:
        if motion_direction in ("in", "out"):
            zoom_direction = motion_direction
        elif motion_direction in ("forward", "reverse"):
            pan_reverse = (motion_direction == "reverse")

    dyn_cap = 1.0 + zoom_rate_per_sec * duration
    zmax    = min(zoom_max, dyn_cap)
    z_target = max(zoom_min, min(zoom_min + (zmax - zoom_min) * (0.25 + 0.50 * u), zmax))

    travel_x = min(max_x, min(int(max_pan_speed * duration), int(max_x * horizontal_travel_ratio)))
    travel_y = min(max_y, min(int(max_pan_speed * duration), int(max_y * vertical_travel_ratio)))
    can_pan  = (travel_y > max(threshold, min_pan_travel)) if is_vertical \
               else (travel_x > max(threshold, min_pan_travel))

    ms                     = max(1.0, float(motion_scale))
    mw, mh                 = int(round(target_w * ms)) , int(round(target_h * ms))
    scaled_w2, scaled_h2   = int(round(scaled_w * ms)), int(round(scaled_h * ms))
    max_x2, max_y2         = max(0, scaled_w2 - mw), max(0, scaled_h2 - mh)
    travel_x2, travel_y2   = int(round(min(max_x2, travel_x * ms))),  int(round(min(max_y2, travel_y * ms)))

    def build_pan_vf():
        if is_vertical  and travel_y2 < 220: return None
        if not is_vertical and travel_x2 < 220: return None

        if is_vertical:
            start_y = int(round(max_y2 * top_bias))
            end_y   = min(start_y + travel_y2, max_y2)
            if pan_reverse: start_y, end_y = end_y, start_y
            x_expr = f"{max_x2 // 2}"
            y_expr = f"trunc({start_y}+({end_y}-{start_y})*{ease_pan})"
        else:
            start_x, end_x = 0, travel_x2
            if pan_reverse: start_x, end_x = end_x, start_x
            x_expr = f"trunc({start_x}+({end_x}-{start_x})*{ease_pan})"
            y_expr = f"{max_y2 // 2}"

        return (
            f"scale={scaled_w2}:{scaled_h2}:flags=lanczos,"
            f"crop={mw}:{mh}:x='{x_expr}':y='{y_expr}',"
            f"scale={target_w}:{target_h}:flags=lanczos,"
            f"setsar=1,setdar=16/9,format=yuv420p"
        )

    def build_zoom_vf():
        if zoom_direction == "out":
            z_expr = f"{z_target}-({z_target}-1.0)*{ease_zoom}"
        else:
            z_expr = f"1.0+({z_target}-1.0)*{ease_zoom}"

        ax = 0.5
        ay = zoom_anchor_vertical_y if is_vertical else zoom_anchor_horizontal_y
        x0 = f"max(0,min(({ax})*iw-ow/2,iw-ow))"
        y0 = f"max(0,min(({ay})*ih-oh/2,ih-oh))"

        return (
            f"scale={scaled_w2}:{scaled_h2}:flags=lanczos,"
            f"zoompan=z='{z_expr}':x='if(eq(on,1),{x0},px)':y='if(eq(on,1),{y0},py)'"
            f":d=1:s={mw}x{mh}:fps={fps},"
            f"scale={target_w}:{target_h}:flags=lanczos,"
            f"setsar=1,setdar=16/9,format=yuv420p"
        )

    if motion_mode == "pan":
        vf = build_pan_vf() or build_zoom_vf()
    elif motion_mode == "zoom":
        vf = build_zoom_vf()
    else:
        vf = (build_pan_vf() or build_zoom_vf()) if (prefer_pan_when_possible and can_pan) \
             else build_zoom_vf()

    vcodec = ["-c:v", "h264_nvenc", "-preset", "p1"] if use_nvenc \
             else ["-c:v", "libx264", "-preset", "slow", "-crf", "18"]

    run_ffmpeg([
        "ffmpeg", "-y",
        "-loop", "1", "-framerate", str(fps), "-t", str(duration), "-i", str(image_path),
        "-vf", vf, "-frames:v", str(total_frames),
        *vcodec, "-pix_fmt", "yuv420p","-fps_mode", "cfr",
        output_path,
    ])

In [30]:
def generate_all_clips(image_files, durations, temp_dir="temp_clips"):
    Path(temp_dir).mkdir(exist_ok=True)

    plans = plan_kenburns_sequence(image_files, durations)

    outputs = []
    for i, (img, dur, plan) in enumerate(zip(image_files, durations, plans)):
        out = f"{temp_dir}/clip_{i}.mp4"

        create_kenburns_clip(
            image_path=img,
            duration=dur,
            output_path=out,
            motion_mode=plan["mode"],
            motion_direction=plan["direction"],
        )
        outputs.append(out)

    return outputs

In [31]:
def distribute_durations_exact(seconds_for_chunk, n_clips, fade):
    target_extra = fade * (n_clips - 1)
    extra_for_each = target_extra / n_clips
    seconds_for_chunk = [s + extra_for_each for s in seconds_for_chunk]
    return seconds_for_chunk

def get_duration(path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())

In [32]:
def concatenate_clips(clips, output_path, fade=0.45):

    durations_local = [get_duration(c) for c in clips]

    cmd = ["ffmpeg", "-y"]
    for c in clips:
        cmd += ["-i", str(c)]

    filter_parts = []

    # IMPORTANT FIX: normalize every clip before xfade
    filter_parts.append(
        "[0:v]fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p[v0]"
    )

    for i in range(1, len(clips)):
        filter_parts.append(
            f"[{i}:v]fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p[v{i}src]"
        )

    cumulative = durations_local[0]
    last = "v0"

    for i in range(1, len(clips)):
        offset = cumulative - fade
        out = f"vx{i}"
        filter_parts.append(
            f"[{last}][v{i}src]xfade=transition=fade:duration={fade}:offset={offset}[{out}]"
        )
        cumulative += durations_local[i] - fade
        last = out

    filter_complex = ";".join(filter_parts)

    cmd += [
        "-safe", "0",
        "-filter_complex", filter_complex,
        "-map", f"[{last}]",
        "-c:v", "h264_nvenc",
        "-preset", "medium",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
      
        str(output_path)
    ]

    run_ffmpeg(cmd)

In [33]:
def add_audio(video_path, audio_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-i", audio_path,
        "-c", "copy",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

def add_subtitles(video_path, srt_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-fflags", "+genpts",
        "-i", video_path,
        "-vf", f"subtitles={srt_path}",
        "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-pix_fmt", "yuv420p",
        "-fps_mode", "cfr",
        "-c:a", "copy",
        output_path
    ]

    run_ffmpeg(cmd)

def cleanup_files():
    temp_dir = "temp_clips"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    temp_dir = "temp_frames"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    for f in os.listdir('Outputs'):
        if f.startswith("combined") or f.startswith("with_audio") or f.startswith("concat_list") or f.startswith("output_subtitles") or f.endswith(".wav"):
            os.remove(os.path.join('Outputs', f))

In [34]:
seconds = distribute_durations_exact(
    seconds_for_chunk,
    n_clips=len(image_files),
    fade=0.45
)

# 1. Generate Ken Burns clips
clips = generate_all_clips(image_files, seconds)

# 2. Concatenate
concatenated = "Outputs/combined.mp4"
concatenate_clips(clips, concatenated)

# 3. Add audio
with_audio = "Outputs/with_audio.mp4"
add_audio(concatenated, "Outputs/Final audio.wav", with_audio)

# 4. Add subtitles
final_output = "Outputs/final_video.mp4"
add_subtitles(with_audio, "Outputs/output_subtitles.srt", final_output)

#cleanup_files()

print("Done:", final_output)

Running: ffmpeg -y -loop 1 -framerate 30 -t 4.7325 -i temp_frames\0000.jpg -vf scale=2688:2008:flags=lanczos,zoompan=z='1.0+(1.0475775-1.0)*(0.5-0.5*cos(PI*on/141))':x='if(eq(on,1),max(0,min((0.5)*iw-ow/2,iw-ow)),px)':y='if(eq(on,1),max(0,min((0.18)*ih-oh/2,ih-oh)),py)':d=1:s=2688x1512:fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p -frames:v 142 -c:v h264_nvenc -preset p1 -pix_fmt yuv420p -fps_mode cfr temp_clips/clip_0.mp4
ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom 

In [35]:
cleanup_files()